# Working with a semantic model in fabric-rlm
The code from the blog post, in order. Set `MODEL` and `WORKSPACE` to your copy of the Olist ecommerce model.
Sections 3 and 4 are each one run (about a minute, and a few minutes for the report).

In [1]:
%pip install -q "fabric-rlm[analytics]==0.6.8"

Note: you may need to restart the kernel to use updated packages.


## 1. Connect

In [2]:
from fabric_rlm import RLM, SemanticModel, FabricLM
from fabric_rlm.semantic_checks import period_bounds

MODEL = "ecommerce_dataset"     # semantic model name or id
WORKSPACE = None                # workspace name or id; None = this notebook's workspace

model = SemanticModel(MODEL, workspace=WORKSPACE)
lm = FabricLM("gpt-5.1", reasoning_effort="high")
model.aggregate(measures=["Total Revenue", "Total Orders"], groupby=["Date[Year]"])

2026-10-02 02:14:43.921: '[__m0]' ('System.Object'/'Object') is serialized as string. To improve performance cast to desired type.
2026-10-02 02:14:43.922: '[__m1]' ('System.Object'/'Object') is serialized as string. To improve performance cast to desired type.


,date_year,total_revenue,total_orders
0,2016,57183.21,312
1,2017,7142672.43,44579
2,2018,8643697.6,53775


## 2. Get the period right

In [3]:
quarters = model.period_coverage("Total Revenue", grain="quarter")
print(quarters)

2026-10-02 02:14:48.822: '[v]' ('System.Object'/'Object') is serialized as string. To improve performance cast to desired type.
Coverage of [Total Revenue] by quarter, dated by 'Date'[Date] (its table is named like a calendar).
As of 2026-10-02 (today). Data from 2016-09-04 to 2018-09-03.
Latest complete quarter: 2018-Q2; latest with data: 2018-Q3.
Trust the data only through 2018-08-26: the data thins out after 2018-08-26: the 7 days to 2018-09-03 average 864 a day against a typical 37,628.
No data in the 33 quarters between 2018-Q3 and the as-of date: the data is stale, so say which period it describes instead of calling it current.
period | dates with data | typical | status
2017-Q1 | 86 | 6.5 | complete
2017-Q2 | 91 | 10 | complete
2017-Q3 | 92 | 48.0 | complete
2017-Q4 | 92 | 86 | complete
2018-Q1 | 90 | 88.5 | complete
2018-Q2 | 91 | 90.5 | complete
2018-Q3 | 61 | 91.0 | partial
2026-Q4 | 0 | 91.0 | in progress


## 3. Ask a question

In [4]:
result = RLM.task(
    """Using the semantic model bound as `model`, which five product categories had the
    highest revenue in 2017, and how much did each earn?""",
    inputs={"model": model},
    outputs={"top_categories": dict, "answer": str},
    skills=["semantic_model"],
    lm=lm,
).run()
print(result.payload["answer"])
print("measures used:", sorted({m for t in result.turns for c in (t.source_calls or []) for m in (c.get("measures") or [])}))

Top 5 product categories by revenue in 2017:
- bed_bath_table: 590,280.44
- health_beauty: 550,420.11
- sports_leisure: 530,730.03
- watches_gifts: 530,086.82
- computers_accessories: 462,761.27
measures used: ['Total Revenue']


In [5]:
result.inspect()

RunInspector(result=RLMResult(submitted=True, payload={'top_categories': {'bed_bath_table': 590280.4400000041, 'health_beauty': 550420.1100000045, 'sports_leisure': 530730.0300000033, 'watches_gifts': 530086.82, 'computers_accessories': 462761.27000000334}, 'answer': 'Top 5 product categories by revenue in 2017:\n- bed_bath_table: 590,280.44\n- health_beauty: 550,420.11\n- sports_leisure: 530,730.03\n- watches_gifts: 530,086.82\n- computers_accessories: 462,761.27'}, trajectory=Trajectory(turns=[TurnRecord(turn=1, code='# First step: inspect the semantic model structure (tables, measures, columns)\nmetadata = model.metadata()\n\nprint("=== Tables ===")\nprint(metadata.tables.to_string(index=False))\n\nprint("\\n=== Measures (first 20) ===")\nprint(metadata.measures.head(20).to_string(index=False))\n\nprint("\\n=== Columns (first 40) ===")\nprint(metadata.columns.head(40).to_string(index=False))', stdout='=== Tables ===\n        table_name                                                                                                                                                                     description  hidden data_category        table_type\n         Customers                                                                                                             Customer dimension. One row per unique customer with location info.   False                           Table\n              Date                                                                                              Calendar date dimension for time-based analysis. Covers the full order date range.   False                           Table\n  Seller Geography                                                                                                     Seller location details: city, state, and lat/long coordinates for mapping.   False                           Table\n          Products                                                                                             Product dimension with categories (English and Portuguese) and physical attributes.   False                           Table\n             Sales                                                                                            Fact table. One row per order line item with pricing, payment, and delivery metrics.   False                           Table\n           Sellers                                                                                                                             Seller dimension with identifier and location code.   False                           Table\nCustomer Geography                                                                                                   Customer location details: city, state, and lat/long coordinates for mapping.   False                Calculated Table\n    Total Measures                                                                                                                                                                                   False                Calculated Table\n  Time Calculation                      Calculation group for time intelligence. Filter by this column to switch any measure between: Current, YTD, Prior Year, or YoY %. Works with all measures.   False               Calculation Group\n      KPI Selector                                                                Disconnected slicer table. No relationships. Select a KPI name and the [Selected KPI] measure returns its value.   False                Calculated Table\n          Currency       Disconnected currency converter. No relationships. Select a currency and [Converted Revenue] shows revenue in that currency. Values: BRL (R$), USD ($), EUR (€), GBP (£).   False                Calculated Table\n          Scenario Disconnected what-if table. No relationships. Select a scenario and [Scenario Revenue] applies the multiplier. Values: Worst Case (0.85x), Base Case (1.0x), Best Case (1.15x).   False                Calculated Table

## 4. A report with a chart, and a few simple checks

In [20]:
import re
import pandas as pd
from fabric_rlm import RLM, FabricLM
from fabric_rlm.semantic_checks import period_bounds

lm = FabricLM("gpt-5.1", reasoning_effort="high")      # use "high" if the reports come out sloppy
quarters = model.period_coverage("Total Revenue", grain="quarter")


def revenue(period, category=None):
    """[Total Revenue] for one quarter: the whole quarter, or one product category. None if nothing matches."""
    start, end, _ = period_bounds(period)
    where = ""
    if category == "Uncategorised":
        where = "ISBLANK(Products[Product Category English]), "
    elif category:
        where = f'Products[Product Category English] = "{category}", '
    value = model.dax(
        f'EVALUATE ROW("v", CALCULATE([Total Revenue], {where}'
        f"'Date'[Date] >= DATE({start.year},{start.month},{start.day}), "
        f"'Date'[Date] < DATE({end.year},{end.month},{end.day})))").iloc[0, 0]
    return None if pd.isna(value) else float(value)


def check_report(payload):
    h = payload["headline"]

    # the report explains itself in words, and has no $ signs
    text = re.sub(r"<[^>]+>", " ", re.sub(r"<svg.*?</svg>|<style.*?</style>", " ", payload["report_html"], flags=re.S))
    assert len(text.split()) >= 250, "Explain each section in plain words: two or three sentences before every chart."
    assert "$" not in text, "Write amounts as BRL, without $ signs."
    assert payload["report_html"].lower().count("<svg") >= 3, "Show the findings as at least three inline SVG charts."

    # complete quarters, and the headline and totals recompute
    for period, cat_key, total_key in ((h["period_from"], "revenue_from", "total_from"),
                                       (h["period_to"], "revenue_to", "total_to")):
        assert quarters.status(period) == "complete", f"{period} is not a complete quarter; use {quarters.latest_complete}."
        for key, actual in ((cat_key, revenue(period, h["category"])), (total_key, revenue(period))):
            assert actual is not None, (
                f'"{h["category"]}" is not a value of Products[Product Category English]. Return the exact '
                'value from the model, such as watches_gifts (use display names only in the report text).')
            assert abs(actual - h[key]) <= 0.005 * actual, f"{key} in {period} should be {actual:,.2f}."


check_report.instructions = (
    "Compare complete quarters only, written like 2018-Q2. headline.category is the exact value from "
    "Products[Product Category English], such as watches_gifts. Headline revenues are the [Total Revenue] measure "
    "for that product category and quarter; the totals are [Total Revenue] for the whole quarter, including "
    "products with no category. Explain every section in words, write amounts as BRL with no $ signs, and show "
    "the findings as at least three inline SVG charts.")

task = """Using the semantic model bound as `model`, find which product category's revenue changed the most,
up or down, between the latest complete quarter and the same quarter a year earlier, and write a polished,
self-contained HTML report for a business reader.

Sections, in this order. Every section starts with two or three sentences of plain-language explanation:
what the chart shows, the main number, and why it matters. Then the chart.
- Executive summary: three KPI cards (the headline category's change in BRL and %, the total revenue change
  across all categories, how many categories grew versus shrank), then a short paragraph on what it means.
- Where it came from: a horizontal bar chart of the five biggest increases and five biggest decreases,
  zero line in the middle, increases and decreases in two colours, category names on the axis and each
  bar's value at the end of the bar.
- Before and after: a dumbbell chart of both quarters for the six biggest movers, labelled with the
  revenue in each quarter (not the change).
- Mix shift: how the share of total revenue moved for the eight largest categories, as paired horizontal
  bars (2017 share next to 2018 share) with the share printed on each bar.
- Insights: three to five short findings, each with the number it rests on. Say what the data shows,
  not causes it cannot show.
- How this was checked: the measure, the two quarters, and that partial quarters were left out.

How to build it: compute the numbers with the model first, then draw every chart with matplotlib
(fig.savefig to an SVG string, embedded inline as <svg>), with a takeaway title such as "Watches & gifts
added BRL 232k, more than any other category", readable labels, and one consistent palette. Write amounts
as "BRL 232,342"; never put a $ sign in the text.

Keep products with no category as their own "Uncategorised" row; do not drop them. The total is [Total Revenue]
for the whole quarter.

Return report_html and headline = {category, period_from, period_to, revenue_from, revenue_to,
change_pct, total_from, total_to}."""

report = RLM.task(task, inputs={"model": model},
                  outputs={"report_html": str, "headline": dict},
                  output_validator=check_report, skills=["semantic_model"], lm=lm, max_turns=24).run()

print("verified:", report.verified, "| turns:", len(report.turns), "| failure:", report.failure_reason)
for t in report.turns:
    for e in t.validation_errors or []:
        print(f"turn {t.turn} sent back: {e[:200]}")

2026-10-02 02:39:21.240: '[v]' ('System.Object'/'Object') is serialized as string. To improve performance cast to desired type.
2026-10-02 02:42:46.848: '[v]' ('System.Object'/'Object') is serialized as string. To improve performance cast to desired type.
2026-10-02 02:42:49.379: '[v]' ('System.Object'/'Object') is serialized as string. To improve performance cast to desired type.
2026-10-02 02:42:51.406: '[v]' ('System.Object'/'Object') is serialized as string. To improve performance cast to desired type.
2026-10-02 02:42:53.233: '[v]' ('System.Object'/'Object') is serialized as string. To improve performance cast to desired type.
verified: True | turns: 8 | failure: None


In [25]:
report.inspect()

RunInspector(result=RLMResult(submitted=True, payload={'report_html': '\n<html>\n<head>\n<meta charset="utf-8">\n<title>Revenue by product category – 2017-Q2 vs 2018-Q2</title>\n<style>\nbody { font-family: Arial, sans-serif; margin: 20px; color: #333; }\nh1 { font-size: 24px; margin-bottom: 5px; }\nh2 { font-size: 18px; margin-top: 30px; margin-bottom: 5px; }\np { font-size: 13px; line-height: 1.5; }\n.chart-container { margin-top: 10px; margin-bottom: 10px; }\nsvg { max-width: 100%; height: auto; }\nul { font-size: 13px; }\n</style>\n</head>\n<body>\n<h1>Revenue shifts by product category: 2017-Q2 to 2018-Q2</h1>\n\n<h2>Executive summary</h2>\n<p>\nThis section summarises the headline movements in revenue between 2017-Q2 and 2018-Q2.\nThe largest shift came from watches_gifts, which added BRL 232,342 in revenue\n(240.9%), while total revenue across all categories increased by\nBRL 1,830,581 (121.9%).\nUnderstanding these top-line changes helps frame where to focus deeper analysis.\n</p>\n<div class="chart-container">\n<svg xmlns:xlink="http://www.w3.org/1999/xlink" width="460.8pt" height="125.28pt" viewBox="0 0 460.8 125.28" xmlns="http://www.w3.org/2000/svg" version="1.1">\n <metadata>\n  <rdf:RDF xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:cc="http://creativecommons.org/ns#" xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#">\n   <cc:Work>\n    <dc:type rdf:resource="http://purl.org/dc/dcmitype/StillImage"/>\n    <dc:date>2026-10-02T02:42:43.443394</dc:date>\n    <dc:format>image/svg+xml</dc:format>\n    <dc:creator>\n     <cc:Agent>\n      <dc:title>Matplotlib v3.10.6, https://matplotlib.org/</dc:title>\n     </cc:Agent>\n    </dc:creator>\n   </cc:Work>\n  </rdf:RDF>\n </metadata>\n <defs>\n  <style type="text/css">*{stroke-linejoin: round; stroke-linecap: butt}</style>\n </defs>\n <g id="figure_1">\n  <g id="patch_1">\n   <path d="M 0 125.28 \nL 460.8 125.28 \nL 460.8 0 \nL 0 0 \nz\n" style="fill: #ffffff"/>\n  </g>\n  <g id="axes_1">\n   <g id="text_1">\n    <!-- watches_gifts in 2018-Q2 -->\n    <g style="fill: #7f7f7f" transform="translate(18.023437 37.540312) scale(0.1 -0.1)">\n     <defs>\n      <path id="DejaVuSans-77" d="M 269 3500 \nL 844 3500 \nL 1563 769 \nL 2278 3500 \nL 2956 3500 \nL 3675 769 \nL 4391 3500 \nL 4966 3500 \nL 4050 0 \nL 3372 0 \nL 2619 2869 \nL 1863 0 \nL 1184 0 \nL 269 3500 \nz\n" transform="scale(0.015625)"/>\n      <path id="DejaVuSans-61" d="M 2194 1759 \nQ 1497 1759 1228 1600 \nQ 959 1441 959 1056 \nQ 959 750 1161 570 \nQ 1363 391 1709 391 \nQ 2188 391 2477 730 \nQ 2766 1069 2766 1631 \nL 2766 1759 \nL 2194 1759 \nz\nM 3341 1997 \nL 3341 0 \nL 2766 0 \nL 2766 531 \nQ 2569 213 2275 61 \nQ 1981 -91 1556 -91 \nQ 1019 -91 701 211 \nQ 384 513 384 1019 \nQ 384 1609 779 1909 \nQ 1175 2209 1959 2209 \nL 2766 2209 \nL 2766 2266 \nQ 2766 2663 2505 2880 \nQ 2244 3097 1772 3097 \nQ 1472 3097 1187 3025 \nQ 903 2953 641 2809 \nL 641 3341 \nQ 956 3463 1253 3523 \nQ 1550 3584 1831 3584 \nQ 2591 3584 2966 3190 \nQ 3341 2797 3341 1997 \nz\n" transform="scale(0.015625)"/>\n      <path id="DejaVuSans-74" d="M 1172 4494 \nL 1172 3500 \nL 2356 3500 \nL 2356 3053 \nL 1172 3053 \nL 1172 1153 \nQ 1172 725 1289 603 \nQ 1406 481 1766 481 \nL 2356 481 \nL 2356 0 \nL 1766 0 \nQ 1100 0 847 248 \nQ 594 497 594 1153 \nL 594 3053 \nL 172 3053 \nL 172 3500 \nL 594 3500 \nL 594 4494 \nL 1172 4494 \nz\n" transform="scale(0.015625)"/>\n      <path id="DejaVuSans-63" d="M 3122 3366 \nL 3122 2828 \nQ 2878 2963 2633 3030 \nQ 2388 3097 2138 3097 \nQ 1578 3097 1268 2742 \nQ 959 2388 959 1747 \nQ 959 1106 1268 751 \nQ 1578 397 2138 397 \nQ 2388 397 2633 464 \nQ 2878 531 3122 666 \nL 3122 134 \nQ 2881 22 2623 -34 \nQ 2366 -91 2075 -91 \nQ 1284 -91 818 406 \nQ 353 903 353 1747 \nQ 353 2603 823 3093 \nQ 1294 3584 2113 3584 \nQ 2378 3584 2631 3529 \nQ 2884 3475 3122 3366 \nz\n" transform="scale(0.015625)"/>\n      <path id="DejaVuSans-68" d="M 3513 2113 \nL 3513 0 \nL 2938 0 \nL 2938 2094 \nQ 2938 2591 2744 2837 \nQ 2550 30

## 5. Look at it, and save it to the Lakehouse

In [26]:
from pathlib import Path
from IPython.display import HTML, display

display(HTML(report.payload["report_html"]))

In [27]:
from pathlib import Path

out = Path("./builtin/category_change.html")
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(report.payload["report_html"], encoding="utf-8")
print("saved", out)

saved builtin/category_change.html
